# 07 — NCBI Taxonomy lookup for the 5,684 core models

This notebook enumerates every `GCF_*.json` file in `data/core_models_kegg2/`, treats each filename's stem as an NCBI RefSeq assembly accession, and queries the **NCBI Datasets v2 REST API** to resolve:

- `organism_name` and `tax_id` via `POST /genome/dataset_report`
- rank-named lineage (`superkingdom`/`kingdom`/`phylum`/`class`/`order`/`family`/`genus`/`species`) via `POST /taxonomy/dataset_report`

The cleaned table is written to `/scratch/ctaylor/core_models_analysis/results/ncbi_taxonomy.csv`.

## KBUtils features used
- **`NotebookSession.for_notebook(...)`** — standard project setup; creates `.kbcache/` with provenance.
- **`session.cache.save / load`** — caches both the raw API responses (`ncbi_taxonomy_raw_v1`, `type_hint='json'`) and the cleaned per-accession DataFrame (`ncbi_taxonomy_v1`, `type_hint='dataframe'`), with `try / except KeyError` to make re-runs cheap.

## API conventions (verified live 2026-06-04)
- Base URL: `https://api.ncbi.nlm.nih.gov/datasets/v2/`
- POST endpoints, not GET (GET path-param has a 100-accession URL limit).
- Chunked at 500 accessions per request with `page_size=1000`.
- API-key header is `api-key: <key>` (optional, read from `NCBI_API_KEY` env-var).
- Rate limits: 5 req/s unkeyed, 10 req/s keyed.
- Unknown/suppressed accessions are **silently dropped** — we compute the set difference to detect them.

Docs: <https://www.ncbi.nlm.nih.gov/datasets/docs/v2/api/rest-api/>, OpenAPI 3.0 at <https://www.ncbi.nlm.nih.gov/datasets/docs/v2/openapi3/openapi3.docs.yaml>.

In [1]:
from pathlib import Path
import sys

# Find the project root (parent of notebooks/) so absolute paths work
# from anywhere the notebook is launched.
PROJECT_ROOT = Path('/scratch/ctaylor/core_models_analysis')
REPORTS = PROJECT_ROOT / 'reports'
RESULTS = PROJECT_ROOT / 'results'
LOGS    = PROJECT_ROOT / 'logs'
DATA    = PROJECT_ROOT / 'data'
SCRIPTS = PROJECT_ROOT / 'scripts'
sys.path.insert(0, str(SCRIPTS))

# KBUtils NotebookSession: opens .kbcache/ alongside this notebook so
# heavy outputs can be saved/loaded with provenance.
from kbutillib.notebook import NotebookSession
session = NotebookSession.for_notebook(notebook_file=__file__ if '__file__' in dir() else None,
                                       project_name='core_models_analysis')
print('Cache directory:', session.kbcache_dir)
print('Notebook name :', session.notebook_name)

Cache directory: /scratch/ctaylor/core_models_analysis/notebooks/.kbcache
Notebook name : None


## 1. Enumerate accessions from `data/core_models_kegg2/`

Each `GCF_*.json` filename stem is treated as a RefSeq assembly accession (e.g. `GCF_000005825.2`).

In [2]:
CORE_MODELS_DIR = DATA / 'core_models_kegg2'

accessions = sorted(p.stem for p in CORE_MODELS_DIR.glob('GCF_*.json'))
print(f'Found {len(accessions):,} GCF_*.json files in {CORE_MODELS_DIR}')
print('First 3:', accessions[:3])
print('Last  3:', accessions[-3:])

Found 5,683 GCF_*.json files in /scratch/ctaylor/core_models_analysis/data/core_models_kegg2
First 3: ['GCF_000005825.2', 'GCF_000005845.2', 'GCF_000006175.1']
Last  3: ['GCF_900478115.1', 'GCF_900478135.1', 'GCF_900478715.1']


## 2. HTTP helper for NCBI Datasets v2

Lightweight wrapper around `requests`. Reuses a single `requests.Session()`, sends `Accept: application/json`, attaches the optional `NCBI_API_KEY` as the `api-key` header, and retries on HTTP 429 / 5xx with exponential backoff.

In [3]:
import os
import time
import json
import requests
from typing import Iterable, Iterator

NCBI_DATASETS_V2 = 'https://api.ncbi.nlm.nih.gov/datasets/v2'
GENOME_REPORT_URL   = f'{NCBI_DATASETS_V2}/genome/dataset_report'
TAXONOMY_REPORT_URL = f'{NCBI_DATASETS_V2}/taxonomy/dataset_report'

# Per docs: 5 rps unkeyed / 10 rps keyed. Sleep > 1/limit to be polite.
API_KEY = os.environ.get('NCBI_API_KEY')
INTER_REQUEST_SLEEP = 0.15 if API_KEY else 0.25
USER_AGENT = 'core_models_analysis/0.1 (NCBI Datasets v2; KBaseUtils notebook 07)'

_HTTP = requests.Session()
_HTTP.headers.update({
    'User-Agent': USER_AGENT,
    'Accept':     'application/json',
    'Content-Type': 'application/json',
})
if API_KEY:
    _HTTP.headers['api-key'] = API_KEY
    print('Using NCBI API key from $NCBI_API_KEY')
else:
    print('No $NCBI_API_KEY set; throttling to 5 req/s (unauthenticated limit).')


def _post_with_retry(url: str, body: dict, *, max_retries: int = 5, base_backoff: float = 1.0) -> dict:
    """POST a JSON body, retrying on HTTP 429 and 5xx with exponential backoff."""
    for attempt in range(max_retries):
        try:
            r = _HTTP.post(url, data=json.dumps(body), timeout=120)
        except requests.RequestException as exc:
            if attempt == max_retries - 1:
                raise
            sleep_s = base_backoff * (2 ** attempt)
            print(f'  transport error {type(exc).__name__}; sleeping {sleep_s:.1f}s then retrying')
            time.sleep(sleep_s)
            continue
        if r.status_code == 200:
            return r.json()
        if r.status_code == 429 or 500 <= r.status_code < 600:
            sleep_s = base_backoff * (2 ** attempt)
            print(f'  HTTP {r.status_code} from {url}; sleeping {sleep_s:.1f}s then retrying')
            time.sleep(sleep_s)
            continue
        # Anything else (400, 401, 403, ...) is fatal.
        r.raise_for_status()
    raise RuntimeError(f'POST {url} failed after {max_retries} retries')


def _chunks(seq, n: int) -> Iterator[list]:
    """Yield successive n-sized chunks from seq."""
    seq = list(seq)
    for i in range(0, len(seq), n):
        yield seq[i:i + n]


def _match_report_to_submitted(report: dict, submitted_set: set[str]) -> str | None:
    """Find which submitted accession (if any) this report corresponds to.

    NCBI may return a report whose `accession` is the *current* GCF when the
    submitted accession was superseded. To recover the submitted->report link,
    check all of: `accession`, `current_accession`, `paired_accession`, and
    versionless prefixes of each (e.g. `GCF_000005825` matches `GCF_000005825.2`).
    Returns the matched submitted accession, or None if no match.
    """
    candidates: list[str] = []
    for key in ('accession', 'current_accession', 'paired_accession'):
        v = report.get(key)
        if v:
            candidates.append(v)
            # Also try the versionless prefix so a submitted GCF_X.1 matches a
            # returned GCF_X.2 (and vice versa).
            if '.' in v:
                candidates.append(v.rsplit('.', 1)[0])
    for c in candidates:
        if c in submitted_set:
            return c
        # Match submitted GCF_X.* against versionless prefix
        for s in submitted_set:
            if s == c or (s.startswith(c + '.') if '.' not in c else False):
                return s
    return None


def fetch_genome_reports(accessions: Iterable[str], *, chunk_size: int = 500) -> list[dict]:
    """POST /genome/dataset_report in chunks; concat all `reports` items.

    Each returned report is annotated in-place with a `_submitted_accession`
    field carrying the user-supplied accession that the report resolved from
    (or None if no submitted accession matched, which would be an API anomaly).
    This lets downstream code key on the submitted accession even when NCBI
    returns the *current* GCF for a superseded input.

    Handles `next_page_token` for completeness (in practice 1 page per chunk).
    """
    accessions = list(accessions)
    all_reports: list[dict] = []
    for ci, chunk in enumerate(_chunks(accessions, chunk_size)):
        chunk_set = set(chunk)
        body = {'accessions': chunk, 'returned_content': 'COMPLETE', 'page_size': 1000}
        page = 0
        chunk_reports: list[dict] = []
        while True:
            resp = _post_with_retry(GENOME_REPORT_URL, body)
            reports = resp.get('reports', []) or []
            chunk_reports.extend(reports)
            tok = resp.get('next_page_token')
            page += 1
            if not tok:
                break
            body = dict(body, page_token=tok)
            time.sleep(INTER_REQUEST_SLEEP)
        # Annotate each report with the submitted accession it matches.
        for rep in chunk_reports:
            rep['_submitted_accession'] = _match_report_to_submitted(rep, chunk_set)
        all_reports.extend(chunk_reports)
        n_matched = sum(1 for rep in chunk_reports if rep.get('_submitted_accession'))
        print(f'  genome chunk {ci+1}: requested={len(chunk)} returned={len(chunk_reports)} '
              f'matched_to_submitted={n_matched} pages={page}')
        time.sleep(INTER_REQUEST_SLEEP)
    return all_reports


def fetch_taxonomy_reports(tax_ids: Iterable[int], *, chunk_size: int = 500) -> list[dict]:
    """POST /taxonomy/dataset_report in chunks; concat all `reports` items."""
    tax_ids = sorted({int(t) for t in tax_ids if t is not None})
    all_reports: list[dict] = []
    for ci, chunk in enumerate(_chunks(tax_ids, chunk_size)):
        # /taxonomy/dataset_report 'taxons' accepts strings (tax_ids, sci names, common names).
        body = {'taxons': [str(t) for t in chunk], 'page_size': 1000}
        page = 0
        while True:
            resp = _post_with_retry(TAXONOMY_REPORT_URL, body)
            reports = resp.get('reports', []) or []
            all_reports.extend(reports)
            tok = resp.get('next_page_token')
            page += 1
            if not tok:
                break
            body = dict(body, page_token=tok)
            time.sleep(INTER_REQUEST_SLEEP)
        print(f'  taxonomy chunk {ci+1}: requested={len(chunk)} returned_so_far={len(all_reports)} pages={page}')
        time.sleep(INTER_REQUEST_SLEEP)
    return all_reports

No $NCBI_API_KEY set; throttling to 5 req/s (unauthenticated limit).


## 3. Sanity test — resolve one known accession

Run this single-accession round-trip before kicking off the full sweep to confirm connectivity, headers, and the response shape we expect. The accession `GCF_000005825.2` is *Bacillus pseudofirmus OF4* (a small bacterium with a stable RefSeq record); it exists in this project's `data/core_models_kegg2/` folder.

In [4]:
SANITY_ACC = 'GCF_000005825.2'

sanity_genome = fetch_genome_reports([SANITY_ACC])
assert len(sanity_genome) == 1, f'expected 1 report, got {len(sanity_genome)}'
_r = sanity_genome[0]
print('accession      :', _r.get('accession'))
print('organism_name  :', _r['organism']['organism_name'])
print('tax_id         :', _r['organism']['tax_id'])
print('assembly_name  :', _r.get('assembly_info', {}).get('assembly_name'))
print('source_database:', _r.get('source_database'))

_tax = fetch_taxonomy_reports([_r['organism']['tax_id']])
assert len(_tax) >= 1
_cls = _tax[0]['taxonomy'].get('classification', {})
print('\nclassification ranks present:', sorted(_cls.keys()))
for rank in ('superkingdom', 'kingdom', 'phylum', 'class', 'order', 'family', 'genus', 'species'):
    print(f'  {rank:13s} = {_cls.get(rank, {}).get("name")}')

  genome chunk 1: requested=1 returned=1 matched_to_submitted=1 pages=1


accession      : GCF_000005825.2
organism_name  : Alkalihalophilus pseudofirmus OF4
tax_id         : 398511
assembly_name  : ASM582v2
source_database: SOURCE_DATABASE_REFSEQ


  taxonomy chunk 1: requested=1 returned_so_far=1 pages=1



classification ranks present: ['class', 'domain', 'family', 'genus', 'kingdom', 'order', 'phylum', 'species']
  superkingdom  = None
  kingdom       = Bacillati
  phylum        = Bacillota
  class         = Bacilli
  order         = Caryophanales
  family        = Bacillaceae
  genus         = Alkalihalophilus
  species       = Alkalihalophilus pseudofirmus


## 4. Fetch genome reports for all accessions (cached)

Cached under `ncbi_taxonomy_raw_v1` with `type_hint='json'`; the cached blob holds the raw genome+taxonomy reports so we can re-derive the cleaned table without re-hitting NCBI.

In [5]:
try:
    raw = session.cache.load('ncbi_taxonomy_raw_v1')
    genome_reports = raw['genome_reports']
    taxonomy_reports = raw['taxonomy_reports']
    print(f'Loaded cached raw responses: {len(genome_reports):,} genome reports, '
          f'{len(taxonomy_reports):,} taxonomy reports')
except KeyError:
    print(f'Fetching genome reports for {len(accessions):,} accessions...')
    genome_reports = fetch_genome_reports(accessions, chunk_size=500)
    print(f'  -> {len(genome_reports):,} reports')

    unique_tax_ids = sorted({r['organism']['tax_id'] for r in genome_reports
                              if r.get('organism', {}).get('tax_id') is not None})
    print(f'Fetching taxonomy reports for {len(unique_tax_ids):,} unique tax_ids...')
    taxonomy_reports = fetch_taxonomy_reports(unique_tax_ids, chunk_size=500)
    print(f'  -> {len(taxonomy_reports):,} reports')

    raw = {'genome_reports': genome_reports, 'taxonomy_reports': taxonomy_reports}
    session.cache.save('ncbi_taxonomy_raw_v1', raw, type_hint='json',
                       metadata={'n_accessions_submitted': len(accessions),
                                 'n_genome_reports': len(genome_reports),
                                 'n_taxonomy_reports': len(taxonomy_reports),
                                 'source': 'NCBI Datasets v2 REST API',
                                 'genome_endpoint': GENOME_REPORT_URL,
                                 'taxonomy_endpoint': TAXONOMY_REPORT_URL})
    print('Saved raw responses under ncbi_taxonomy_raw_v1')

Fetching genome reports for 5,683 accessions...


  genome chunk 1: requested=500 returned=474 matched_to_submitted=474 pages=1


  genome chunk 2: requested=500 returned=473 matched_to_submitted=473 pages=1


  genome chunk 3: requested=500 returned=480 matched_to_submitted=480 pages=1


  genome chunk 4: requested=500 returned=496 matched_to_submitted=496 pages=1


...

  taxonomy chunk 9: requested=500 returned_so_far=4500 pages=1


  taxonomy chunk 10: requested=500 returned_so_far=5000 pages=1


  taxonomy chunk 11: requested=351 returned_so_far=5351 pages=1


  -> 5,351 reports


Saved raw responses under ncbi_taxonomy_raw_v1


## 5. Build the cleaned per-accession DataFrame

Columns: `assembly_accession, organism_name, tax_id, assembly_name, status, superkingdom, kingdom, phylum, class, order, family, genus, species`.

Accessions that NCBI silently dropped (suppressed/replaced/unknown) get one row each with `status='missing'` and `NaN` for every other column.

In [6]:
import pandas as pd
import numpy as np

RANKS = ('superkingdom', 'kingdom', 'phylum', 'class', 'order', 'family', 'genus', 'species')

# Index taxonomy reports by tax_id (int) for fast lookup.
tax_by_id: dict[int, dict] = {}
for tr in taxonomy_reports:
    tid = tr.get('taxonomy', {}).get('tax_id')
    if tid is not None:
        tax_by_id[int(tid)] = tr


def _classification_row(tax_id: int | None) -> dict:
    """Return {rank: name or None} for the 8 standard ranks. NOTE: 'class' is a
    Python keyword — we access it via dict indexing."""
    out = {rank: None for rank in RANKS}
    if tax_id is None:
        return out
    tr = tax_by_id.get(int(tax_id))
    if tr is None:
        return out
    cls = tr.get('taxonomy', {}).get('classification', {}) or {}
    for rank in RANKS:
        node = cls.get(rank) or {}
        out[rank] = node.get('name')
    # 'domain' supersedes 'superkingdom' in NCBI Taxonomy as of ~2024; backfill if needed.
    if out['superkingdom'] is None:
        dom = cls.get('domain') or {}
        if dom.get('name'):
            out['superkingdom'] = dom['name']
    return out


# Build rows keyed on the SUBMITTED accession (not the returned `accession`,
# which may be the *current* GCF for a superseded input). The fetch step
# annotated each report with `_submitted_accession`; we use that here so the
# `assembly_accession` column matches the source `GCF_*.json` filename and
# downstream joins are reliable. `current_accession` is kept as a separate
# column to surface NCBI's revision when present.
submitted = set(accessions)
rows: list[dict] = []
matched_submitted: set[str] = set()
unmatched_reports = 0
for r in genome_reports:
    sub_acc = r.get('_submitted_accession')
    if sub_acc is None:
        # API anomaly: report did not match any submitted accession in its
        # chunk. For backward compatibility we still emit it keyed on the
        # returned accession, but it will not satisfy any submitted accession
        # and will leave the corresponding submitted entry in `missing`.
        unmatched_reports += 1
        sub_acc = r.get('accession')
    else:
        matched_submitted.add(sub_acc)
    org = r.get('organism', {}) or {}
    tax_id = org.get('tax_id')
    row = {
        'assembly_accession': sub_acc,
        'current_accession':  r.get('current_accession') or r.get('accession'),
        'organism_name':      org.get('organism_name'),
        'tax_id':             tax_id,
        'assembly_name':      (r.get('assembly_info') or {}).get('assembly_name'),
        'status':             'resolved',
    }
    row.update(_classification_row(tax_id))
    rows.append(row)

if unmatched_reports:
    print(f'WARNING: {unmatched_reports} returned report(s) could not be matched to '
          f'any submitted accession; their `assembly_accession` falls back to the '
          f'returned `accession` value.')

# Missing accessions = submitted minus those we matched a report to.
missing = sorted(submitted - matched_submitted)
for acc in missing:
    row = {
        'assembly_accession': acc,
        'current_accession':  None,
        'organism_name':      None,
        'tax_id':             None,
        'assembly_name':      None,
        'status':             'missing',
    }
    row.update({rank: None for rank in RANKS})
    rows.append(row)

df = pd.DataFrame(rows, columns=[
    'assembly_accession', 'current_accession', 'organism_name', 'tax_id',
    'assembly_name', 'status', *RANKS,
])
df = df.sort_values('assembly_accession').reset_index(drop=True)
# tax_id as nullable Int64 (NaN-safe).
df['tax_id'] = pd.array(df['tax_id'], dtype='Int64')

print(f'Rows in cleaned table: {len(df):,}')
print(f'  resolved: {(df.status == "resolved").sum():,}')
print(f'  missing : {(df.status == "missing").sum():,}')
df.head()

Rows in cleaned table: 5,683
  resolved: 5,561
  missing : 122


,assembly_accession,current_accession,organism_name,tax_id,assembly_name,status,superkingdom,kingdom,phylum,class,order,family,genus,species
0,GCF_000005825.2,GCF_000005825.2,Alkalihalophilus pseudofirmus OF4,398511,ASM582v2,resolved,Bacteria,Bacillati,Bacillota,Bacilli,Caryophanales,Bacillaceae,Alkalihalophilus,Alkalihalophilus pseudofirmus
1,GCF_000005845.2,GCF_000005845.2,Escherichia coli str. K-12 substr. MG1655,511145,ASM584v2,resolved,Bacteria,Pseudomonadati,Pseudomonadota,Gammaproteobacteria,Enterobacterales,Enterobacteriaceae,Escherichia,Escherichia coli
2,GCF_000006175.1,None,None,<NA>,None,missing,None,None,None,None,None,None,None,None
3,GCF_000006605.1,GCF_000006605.1,Corynebacterium jeikeium K411,306537,ASM660v1,resolved,Bacteria,Bacillati,Actinomycetota,Actinomycetes,Mycobacteriales,Corynebacteriaceae,Corynebacterium,Corynebacterium jeikeium
4,GCF_000006625.1,GCF_000006625.1,Ureaplasma parvum serovar 3 str. ATCC 700970,273119,ASM662v1,resolved,Bacteria,Bacillati,Mycoplasmatota,None,Mycoplasmoidales,Mycoplasmoidaceae,Ureaplasma,Ureaplasma parvum


## 6. Cache the cleaned DataFrame and write CSV/JSON outputs

In [ ]:
try:
    df = session.cache.load('ncbi_taxonomy_v1')
    print(f'Loaded cached cleaned frame: {len(df):,} rows')
except KeyError:
    session.cache.save('ncbi_taxonomy_v1', df, type_hint='dataframe',
                       metadata={'n_rows': len(df),
                                 'n_resolved': int((df.status == 'resolved').sum()),
                                 'n_missing':  int((df.status == 'missing').sum()),
                                 'source': 'NCBI Datasets v2 (genome + taxonomy /dataset_report)',
                                 'columns': list(df.columns)})
    print(f'Saved cleaned frame as ncbi_taxonomy_v1 ({len(df):,} rows)')

In [ ]:
# Always (re-)write the human-readable outputs. Guard on df being defined so
# running this cell alone after a kernel restart fails fast with a clear error.
if 'df' not in dir():
    raise NameError("`df` is not defined; run the previous cells (or the cache-load cell) first.")

CSV_OUT  = RESULTS / 'ncbi_taxonomy.csv'
JSON_OUT = RESULTS / 'ncbi_taxonomy.json'
df.to_csv(CSV_OUT, index=False)
df.to_json(JSON_OUT, orient='records', indent=2)
print(f'Wrote {CSV_OUT}')
print(f'Wrote {JSON_OUT}')

## 7. Summary statistics

In [ ]:
n_total    = len(df)
n_resolved = int((df.status == 'resolved').sum())
n_missing  = int((df.status == 'missing').sum())

print(f'Total accessions submitted : {n_total:,}')
print(f'Resolved by NCBI            : {n_resolved:,}')
print(f'Missing / suppressed / unknown: {n_missing:,}')
if n_resolved:
    print(f'Distinct tax_ids            : {df.loc[df.status == "resolved", "tax_id"].nunique():,}')
    print(f'Distinct genera             : {df.loc[df.status == "resolved", "genus"].nunique():,}')
    print(f'Distinct species            : {df.loc[df.status == "resolved", "species"].nunique():,}')

print('\nTop 10 superkingdoms by count:')
print(df.loc[df.status == 'resolved', 'superkingdom'].value_counts(dropna=False).head(10).to_string())

print('\nTop 10 phyla by count:')
print(df.loc[df.status == 'resolved', 'phylum'].value_counts(dropna=False).head(10).to_string())

print('\nTop 10 genera by count:')
print(df.loc[df.status == 'resolved', 'genus'].value_counts(dropna=False).head(10).to_string())

print('\nTop 10 species by count:')
print(df.loc[df.status == 'resolved', 'species'].value_counts(dropna=False).head(10).to_string())

if n_missing:
    print(f'\nFirst 10 missing accessions:')
    for a in df.loc[df.status == 'missing', 'assembly_accession'].head(10):
        print(f'  {a}')

## Report: `reports/ncbi_taxonomy.md`

**Purpose.** Resolve NCBI organism + lineage metadata for every RefSeq assembly accession present in `data/core_models_kegg2/`. Output feeds downstream taxonomy-aware analyses (panel diversification, per-clade growth/gap summaries, etc.).

**Approach.** Two POST calls to the NCBI Datasets v2 REST API:

1. `POST /genome/dataset_report` → `organism_name`, `tax_id`, `assembly_name` per accession (chunks of 500).
2. `POST /taxonomy/dataset_report` → rank-named lineage (`superkingdom`/`kingdom`/`phylum`/`class`/`order`/`family`/`genus`/`species`) per unique `tax_id`.

Raw responses are cached under `ncbi_taxonomy_raw_v1` (json) and the cleaned table under `ncbi_taxonomy_v1` (dataframe). Re-runs that find the cache skip all network I/O.

**Outputs.**
- `results/ncbi_taxonomy.csv` — one row per submitted accession, with `status='resolved'` or `status='missing'`.
- `results/ncbi_taxonomy.json` — same content, JSON records orientation.

**Caveats.**
- NCBI silently drops unknown / suppressed / superseded accessions. We surface them as `status='missing'` rows by set-differencing submitted vs. returned accessions (including `current_accession` and `paired_accession`).
- `superkingdom` is deprecated in favour of `domain` in NCBI Taxonomy (≈2024). We fall back to `domain` when `superkingdom` is absent.
- Unranked / informal taxa (`environmental samples`, `metagenome`) may have empty `classification` blocks — their lineage cells will be `NaN`.
- Set the `NCBI_API_KEY` env-var to raise the rate limit from 5 → 10 req/s. The full sweep is < 30 POSTs either way.